# Module 12: Interactive CPython Internals — AST, Bytecode, Refcounts & GC

Welcome to **Module 12**! In this laboratory, you will look beneath the Python syntax and examine the CPython virtual machine:
1. **From Source to Abstract Syntax Tree (AST):** Parsing and visualizing the AST.
2. **Bytecode Disassembly with `dis`:** Inspecting stack evaluation and opcodes (`LOAD_FAST`, `BINARY_OP`, `RETURN_VALUE`).
3. **Code Object Anatomy:** Examining `__code__` attributes (`co_varnames`, `co_consts`, `co_code`).
4. **CPython Memory & Reference Counting:** Inspecting `sys.getrefcount()` and object headers.
5. **Garbage Collection Cycles:** Creating reference cycles and forcing `gc.collect()`.
6. **Memory Optimization with `__slots__`:** Measuring the 65%+ RAM savings vs standard `__dict__`.
7. **Interactive Challenge:** Building a static AST security linter that detects dangerous `eval()` calls.


## 1. Abstract Syntax Tree (AST) Generation


In [ ]:
import ast

code_sample = """
def compute_interest(principal: float, rate: float) -> float:
    return principal * (1.0 + rate)
"""

tree = ast.parse(code_sample)
print("Root AST Node:", type(tree).__name__)
print("Function definition node:", tree.body[0].name)
print("\nFormatted AST Structure:")
print(ast.dump(tree, indent=2))


## 2. Dissecting Bytecode with `dis`


In [ ]:
import dis


def add_numbers(x: int, y: int) -> int:
    return x + y

print("Disassembly of add_numbers:")
dis.dis(add_numbers)

# Inspecting the underlying code object:
co = add_numbers.__code__
print("\nCode Object Inspection:")
print("  co_name:     ", co.co_name)
print("  co_argcount: ", co.co_argcount)
print("  co_varnames: ", co.co_varnames)
print("  co_consts:   ", co.co_consts)
print("  co_code (raw bytes):", co.co_code)


## 3. Reference Counting & Object Identity


In [ ]:
import sys

# In CPython, every object begins with PyObject_HEAD containing ob_refcnt and ob_type!
sample_list = [10, 20, 30]
print("Initial refcount:", sys.getrefcount(sample_list))  # Note: getrefcount adds 1 temporary ref!

alias_1 = sample_list
print("After 1 alias:", sys.getrefcount(sample_list))

alias_2 = sample_list
print("After 2 aliases:", sys.getrefcount(sample_list))

del alias_1
print("After deleting alias_1:", sys.getrefcount(sample_list))


## 4. The Small Integer Cache Singleton Pool


In [ ]:
# CPython pre-allocates small integer singletons from -5 to 256 for instant lookup:
x = 250
y = 250
print("250 is 250:", x is y)  # True: points to identical memory address!
print("id(x) == id(y):", id(x) == id(y))

# Numbers outside the pre-allocated cache create separate objects in standard execution:
a = 100_000
b = int("100000")
print("100000 is 100000:", a is b)  # False (different objects in memory)
print("100000 == 100000:", a == b)  # True (equal values)


## 5. Garbage Collection & Cyclic References


In [ ]:
import gc


class Node:
    def __init__(self, val):
        self.val = val
        self.link = None

# Creating an isolated reference cycle:
n1 = Node(1)
n2 = Node(2)
n1.link = n2
n2.link = n1

# Delete external references:
del n1
del n2

# Refcounts alone CANNOT reclaim n1 and n2 because they reference each other!
# The CPython generational garbage collector detects unreachable cycles:
unreachable = gc.collect()
print(f"Garbage collector ran and collected {unreachable} unreachable cyclic objects!")


## 6. Memory Optimization: `__slots__` vs `__dict__`


In [ ]:
class StandardObject:
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c

class SlottedObject:
    __slots__ = ('a', 'b', 'c')
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c

std_obj = StandardObject(1, 2, 3)
slotted_obj = SlottedObject(1, 2, 3)

print("Standard object has dynamic dict:", hasattr(std_obj, '__dict__'))
print("Slotted object has NO dict:", hasattr(slotted_obj, '__dict__'))

std_size = sys.getsizeof(std_obj) + sys.getsizeof(std_obj.__dict__)
slotted_size = sys.getsizeof(slotted_obj)

print(f"Standard instance memory: {std_size} bytes")
print(f"Slotted instance memory:  {slotted_size} bytes")
print(f"RAM reduction per object: {((std_size - slotted_size)/std_size)*100:.1f}% savings!")


## 7. Interactive Challenge: Build an AST Security Linter


In [ ]:
# CHALLENGE: Build an AST NodeVisitor that inspects Python source code
# and flags any dangerous calls to eval() or exec()!

class SecurityLinter(ast.NodeVisitor):
    def __init__(self):
        self.violations = []

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name) and node.func.id in {"eval", "exec"}:
            self.violations.append(f"Forbidden dynamic execution: '{node.func.id}()' at line {node.lineno}")
        self.generic_visit(node)

test_code = """
def safe_math(a, b):
    return a + b

def dangerous_function(user_input):
    return eval(user_input)
"""

linter = SecurityLinter()
linter.visit(ast.parse(test_code))

print("Detected Security Violations:")
for v in linter.violations:
    print(f"  [ALERT] {v}")

assert len(linter.violations) == 1
assert "eval" in linter.violations[0]
print("[ALL INTERNALS CHALLENGES PASSED] AST Linter operational!")
